# Lunar Crater Detection — Automatic NASA Image Inference (GeoTIFF/Rasterio Updated)

Kaggle notebook for lunar crater detection on GeoTIFF/TIFF, PNG, JPG, NPY, NPZ and FITS imagery. Add the crater checkpoint as a Kaggle Input and attach/upload one or more images. No manual image conversion, stretching, resizing, cropping or annotation is required.

## Key features
- Reads TIFF/GeoTIFF using **Rasterio**, avoiding OpenCV's unsupported GeoTIFF-tag warnings.
- Reads the first raster band by default and records CRS, transform, bounds, resolution, NoData and band metadata.
- Converts documented NoData pixels to NaN so they are excluded from percentile stretching; detections centred on NoData pixels are discarded.
- Uses training-matched 256 × 256 source tiles plus 512 × 512 context tiles; maps predictions to original image coordinates.
- Uses per-tile 2nd–98th percentile stretching, coordinate-safe edge padding and scale-aware duplicate merging.
- Exports candidate tables, threshold reports, annotated images, geospatial metadata, run metadata and a ZIP archive.

**Threshold note:** 0.55 is the validation-selected operating point. 0.15 is exploratory only. More detections at a lower threshold do not prove higher recall without ground truth.

## 1. Install and import dependencies
Run this cell first. Internet is needed only if Ultralytics or Astropy is not already installed.

In [11]:
%pip -q install -U "ultralytics>=8.3.0" opencv-python-headless pandas matplotlib tqdm astropy rasterio

import os, json, math, warnings, platform, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import rasterio
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO
from IPython.display import display

try:
    import torch
except Exception:
    torch = None

print("Python:", platform.python_version())
print("OpenCV:", cv2.__version__)
print("Rasterio:", rasterio.__version__)
print("GPU available:", bool(torch is not None and torch.cuda.is_available()))
if torch is not None and torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Note: you may need to restart the kernel to use updated packages.
Python: 3.13.16
OpenCV: 5.1.0
Rasterio: 1.5.2
GPU available: True
GPU: Tesla T4


## 2. Configuration and automatic checkpoint discovery
Attach the model dataset using Kaggle **Add Input**. The notebook prefers the fine-tuned checkpoint, then falls back to the fresh baseline. If multiple copies exist, it stops rather than silently choosing the wrong one.

In [12]:
# Leave blank for automatic discovery. Set this only if you need a specific checkpoint.
MODEL_PATH = ""

# Image input: upload files to /kaggle/working/exam_images, OR set this to an attached Kaggle dataset folder.
INPUT_DIR = Path("/kaggle/input/datasets/n1zamudd1n/test-crater")

# Training/inference configuration
IMAGE_SIZE = 640                 # YOLO imgsz used during training
SOURCE_TILE_SIZES = [256, 512]   # 256 is training-matched; 512 is context-scale diagnostic
TILE_OVERLAP = 0.25
LOW_CONFIDENCE_CACHE = 0.001     # preserve candidates for later review
VALIDATION_CONFIDENCE = 0.55     # validation-selected operating point
DISPLAY_CONFIDENCE = 0.15        # exploratory only; not validation-calibrated
THRESHOLDS_TO_REPORT = [0.55, 0.35, 0.25, 0.15, 0.10, 0.05]
MAX_DETECTIONS_PER_TILE = 1000
MODEL_IOU = 0.70

OUTPUT_DIR = Path("/kaggle/working/crater_detection_results")
ANNOTATED_DIR = OUTPUT_DIR / "annotated_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH:
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    def find_named(filename):
        found=[]
        for root in roots:
            if root.exists():
                found.extend(root.rglob(filename))
        return sorted(set(p.resolve() for p in found if p.is_file()))

    fine = find_named("best_finetuned_crater_model.pt")
    base = find_named("best_crater_model_fresh_v2.pt")
    if len(fine) == 1:
        MODEL_PATH = str(fine[0]); MODEL_TYPE = "fine-tuned"
    elif len(fine) > 1:
        raise RuntimeError("Multiple fine-tuned checkpoints found. Set MODEL_PATH explicitly:\n" + "\n".join(map(str,fine)))
    elif len(base) == 1:
        MODEL_PATH = str(base[0]); MODEL_TYPE = "fresh baseline fallback"
        print("WARNING: fine-tuned checkpoint not found; using baseline.")
    elif len(base) > 1:
        raise RuntimeError("Multiple baseline checkpoints found. Set MODEL_PATH explicitly:\n" + "\n".join(map(str,base)))
    else:
        raise FileNotFoundError("No crater checkpoint found. Attach best_finetuned_crater_model.pt (preferred) or best_crater_model_fresh_v2.pt as a Kaggle Input, or set MODEL_PATH.")
else:
    MODEL_TYPE = "manually selected"

MODEL_PATH = Path(MODEL_PATH)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}")

model = YOLO(str(MODEL_PATH))
print("Model type:", MODEL_TYPE)
print("Checkpoint:", MODEL_PATH)
print("Classes:", model.names)
# A class label such as 'item' is checkpoint metadata; it does not stop inference.
# For this project, confirm the checkpoint has exactly one detection class.
if isinstance(model.names, dict) and len(model.names) != 1:
    print("WARNING: Expected a single crater class, but checkpoint exposes", len(model.names), "classes.")
else:
    print("Detection class label:", list(model.names.values())[0] if isinstance(model.names, dict) else model.names)
    print("NOTE: If the label is 'item', verify the training dataset YAML used that name. The model can still detect craters; the label itself does not change its learned weights.")
print("Source tile scales:", SOURCE_TILE_SIZES, "-> YOLO imgsz:", IMAGE_SIZE)
print("Validation threshold:", VALIDATION_CONFIDENCE, "| exploratory display:", DISPLAY_CONFIDENCE)

Model type: fine-tuned
Checkpoint: /kaggle/input/datasets/latebloomer18/bcdm9999/best_finetuned_crater_model.pt
Classes: {0: 'item'}
Detection class label: item
NOTE: If the label is 'item', verify the training dataset YAML used that name. The model can still detect craters; the label itself does not change its learned weights.
Source tile scales: [256, 512] -> YOLO imgsz: 640
Validation threshold: 0.55 | exploratory display: 0.15


## 3. Find input images automatically
Preferred location is `/kaggle/working/exam_images`. If it is empty, the notebook searches `/kaggle/input` recursively for supported image files. If your images are in an attached Kaggle dataset, this avoids having to edit a path.

In [13]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".geotiff", ".webp", ".npy", ".npz", ".fits", ".fit", ".fts"}
INPUT_DIR.mkdir(parents=True, exist_ok=True)

def discover_images():
    local = sorted(p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS)
    if local:
        return local, str(INPUT_DIR)
    kaggle_root = Path("/kaggle/input")
    found=[]
    if kaggle_root.exists():
        for p in kaggle_root.rglob("*"):
            if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS:
                # Avoid accidentally treating model/checkpoint files as input imagery.
                found.append(p)
    return sorted(found), "/kaggle/input (automatic search)"

image_paths, discovered_from = discover_images()
print("Image search location:", discovered_from)
print("Images found:", len(image_paths))
for p in image_paths[:50]: print(p)
if not image_paths:
    raise FileNotFoundError("No images found. Upload images to /kaggle/working/exam_images or attach an image dataset using Kaggle Add Input.")

Image search location: /kaggle/input/datasets/n1zamudd1n/test-crater
Images found: 1
/kaggle/input/datasets/n1zamudd1n/test-crater/nac_crop_georef.tif


## 4. Load and standardize images automatically

GeoTIFF/TIFF files are opened with Rasterio rather than `cv2.imread`. This is important because OpenCV may warn that GeoTIFF tags are unknown; it can still read pixel data, but it does not provide a geospatially aware metadata workflow. Rasterio reads band data and captures CRS, affine transform, bounds, pixel resolution and NoData metadata.

The default for multi-band TIFFs is **band 1**. Change `TIFF_BAND` below if your product stores the desired grayscale image in another band. Documented NoData values and non-finite pixels become NaN and are excluded from contrast stretching. Ordinary PNG/JPG zero values are not automatically treated as NoData.

In [14]:
# TIFF band to use for multi-band TIFF/GeoTIFF products (Rasterio bands are 1-based).
TIFF_BAND = 1

# Optional override. Leave as None to use the TIFF's own NoData metadata.
# Only set this when the product documentation explicitly specifies a NoData value.
TIFF_NODATA_OVERRIDE = None

# Metadata for every loaded source image is stored here and exported later.
GEOSPATIAL_METADATA = {}
TILE_DIAGNOSTICS = {}


def load_grayscale_image(path):
    """Load a grayscale image without losing scientific bit depth or GeoTIFF metadata."""
    path = Path(path)
    ext = path.suffix.lower()

    if ext in {".tif", ".tiff", ".geotiff"}:
        with rasterio.open(path) as src:
            if src.count < TIFF_BAND or TIFF_BAND < 1:
                raise ValueError(
                    f"Requested TIFF_BAND={TIFF_BAND}, but raster has {src.count} band(s)."
                )
            band = src.read(TIFF_BAND, masked=True)
            arr = np.asarray(band.astype("float32").filled(np.nan), dtype=np.float32)
            nodata_value = TIFF_NODATA_OVERRIDE if TIFF_NODATA_OVERRIDE is not None else src.nodata

            # Also mask an explicitly documented override if the raster's own mask did not.
            if nodata_value is not None:
                arr[arr == float(nodata_value)] = np.nan

            GEOSPATIAL_METADATA[str(path)] = {
                "driver": src.driver,
                "width_px": int(src.width),
                "height_px": int(src.height),
                "band_count": int(src.count),
                "selected_band_1based": int(TIFF_BAND),
                "dtype": str(src.dtypes[TIFF_BAND - 1]),
                "crs": str(src.crs) if src.crs else None,
                "transform": tuple(src.transform),
                "bounds": tuple(src.bounds),
                "resolution": tuple(src.res),
                "nodata": nodata_value,
                "valid_pixel_count": int(np.isfinite(arr).sum()),
            }

    elif ext == ".npy":
        arr = np.load(path, allow_pickle=False)
    elif ext == ".npz":
        with np.load(path, allow_pickle=False) as z:
            keys = list(z.keys())
            if len(keys) != 1:
                raise ValueError(f"NPZ must contain exactly one array; found {keys}")
            arr = z[keys[0]]
    elif ext in {".fits", ".fit", ".fts"}:
        from astropy.io import fits
        with fits.open(path, memmap=False) as hdul:
            arr = next((h.data for h in hdul if h.data is not None), None)
        if arr is None:
            raise ValueError(f"No image array found in FITS: {path}")
    else:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if arr is None:
            raise ValueError(f"Could not read image: {path}")
        if arr.ndim == 3:
            if arr.shape[2] == 4:
                arr = cv2.cvtColor(arr, cv2.COLOR_BGRA2GRAY)
            elif arr.shape[2] == 3:
                arr = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
            else:
                raise ValueError(f"Unsupported channel count {arr.shape[2]} in {path}")

    arr = np.asarray(arr)
    while arr.ndim > 2 and 1 in arr.shape:
        arr = np.squeeze(arr)
    if arr.ndim == 3 and arr.shape[-1] in (3, 4):
        arr = cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    if arr.ndim != 2:
        raise ValueError(
            f"Expected a 2D grayscale image after channel handling; got {arr.shape}. "
            "For multi-band TIFFs, select the required TIFF_BAND."
        )

    # Keep float data when necessary so NaN can represent invalid/NoData pixels.
    if np.issubdtype(arr.dtype, np.floating):
        arr = arr.astype(np.float32, copy=False)
    return arr


def stretch_training_style(arr, lo=2, hi=98):
    arr = np.asarray(arr)
    valid = np.isfinite(arr)
    out = np.zeros(arr.shape, dtype=np.uint8)
    if valid.sum() < 10:
        raise ValueError(
            f"Image/tile has only {int(valid.sum())} finite pixels; "
            "at least 10 are required for percentile stretching."
        )
    vals = arr[valid].astype(np.float32)
    low, high = np.percentile(vals, [lo, hi])
    if not np.isfinite(low + high) or high <= low:
        low, high = float(vals.min()), float(vals.max())
    if high <= low:
        out[valid] = 127
    else:
        out[valid] = np.clip(
            (arr[valid].astype(np.float32) - low) * 255.0 / (high - low),
            0, 255
        ).astype(np.uint8)
    return out, float(low), float(high)

print("Rasterio-aware image loader and training-style stretch are ready.")

Rasterio-aware image loader and training-style stretch are ready.


## 5. Training-matched 256-pixel tiling and coordinate-safe inference
This is the main correction. The previous inference used 640 × 640 source crops, while SomBench training images were 256 × 256 and were then letterboxed/upscaled by YOLO to 640. The notebook now runs both 256 × 256 (training-matched primary scale) and 512 × 512 (context diagnostic scale). All returned boxes are mapped from each tile's original coordinate system to the full image. Compare per-scale counts before treating the combined output as a final result.

In [15]:
# Tile-generation helpers (required by predict_image_tiles).
def tile_starts(length, tile_size, overlap):
    """Return deterministic tile origins that cover the full image, including edges."""
    length = int(length)
    tile_size = int(tile_size)
    if length <= 0 or tile_size <= 0:
        raise ValueError(f"Invalid image/tile dimension: length={length}, tile_size={tile_size}")
    if not 0 <= float(overlap) < 1:
        raise ValueError(f"TILE_OVERLAP must be in [0, 1); got {overlap}")
    if length <= tile_size:
        return [0]
    stride = max(1, int(round(tile_size * (1.0 - float(overlap)))))
    starts = list(range(0, length - tile_size + 1, stride))
    last_start = length - tile_size
    if starts[-1] != last_start:
        starts.append(last_start)
    return starts


def extract_fixed_tile(image, x0, y0, tile_size):
    """Extract a tile and NaN-pad right/bottom edges; return tile and real valid width/height."""
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError(f"Expected 2D grayscale image, got shape {image.shape}")
    h, w = image.shape
    x0, y0, tile_size = int(x0), int(y0), int(tile_size)
    if x0 < 0 or y0 < 0 or x0 >= w or y0 >= h:
        raise ValueError(f"Tile origin outside image: ({x0}, {y0}) for image {w}x{h}")
    valid_w = min(tile_size, w - x0)
    valid_h = min(tile_size, h - y0)
    tile = np.full((tile_size, tile_size), np.nan, dtype=np.float32)
    crop = image[y0:y0 + valid_h, x0:x0 + valid_w]
    tile[:valid_h, :valid_w] = crop.astype(np.float32, copy=False)
    return tile, valid_w, valid_h


def predict_image_tiles(gray, image_name):
    h, w = gray.shape
    rows = []
    total_tiles = 0
    skipped_sparse_tiles = 0
    valid_image_mask = np.isfinite(gray)
    image_valid_count = int(valid_image_mask.sum())
    if image_valid_count < 10:
        raise ValueError(
            f"Source image has only {image_valid_count} finite pixels out of {gray.size}. "
            "This points to a TIFF band/NoData mask issue; inspect the TIFF diagnostics "
            "before running inference."
        )

    for tile_size in SOURCE_TILE_SIZES:
        xs = tile_starts(w, tile_size, TILE_OVERLAP)
        ys = tile_starts(h, tile_size, TILE_OVERLAP)
        tile_number = 0
        for y0 in ys:
            for x0 in xs:
                tile_number += 1
                total_tiles += 1
                tile, valid_w, valid_h = extract_fixed_tile(gray, x0, y0, tile_size)
                # Edge/NoData-heavy tiles can contain too few real pixels for a meaningful
                # percentile stretch. Skip them; do not abort processing of the whole TIFF.
                tile_valid_count = int(np.isfinite(tile[:valid_h, :valid_w]).sum())
                if tile_valid_count < 10:
                    skipped_sparse_tiles += 1
                    continue
                tile_stretched, stretch_low, stretch_high = stretch_training_style(tile)
                tile_rgb = cv2.cvtColor(tile_stretched, cv2.COLOR_GRAY2RGB)
                result = model.predict(
                    source=tile_rgb,
                    conf=LOW_CONFIDENCE_CACHE,
                    imgsz=IMAGE_SIZE,
                    max_det=MAX_DETECTIONS_PER_TILE,
                    iou=MODEL_IOU,
                    verbose=False
                )[0]
                if result.boxes is None:
                    continue

                for box in result.boxes:
                    bx1, by1, bx2, by2 = map(float, box.xyxy[0].cpu().numpy())
                    score = float(box.conf[0].cpu().item())
                    bx1 = min(max(bx1, 0.0), float(valid_w))
                    bx2 = min(max(bx2, 0.0), float(valid_w))
                    by1 = min(max(by1, 0.0), float(valid_h))
                    by2 = min(max(by2, 0.0), float(valid_h))
                    if bx2 <= bx1 or by2 <= by1:
                        continue
                    cx_local, cy_local = (bx1 + bx2) / 2, (by1 + by2) / 2
                    if cx_local >= valid_w or cy_local >= valid_h:
                        continue

                    # Exclude boxes whose centre lands on a documented NoData/non-finite pixel.
                    gx_center = min(w - 1, max(0, int(round(x0 + cx_local))))
                    gy_center = min(h - 1, max(0, int(round(y0 + cy_local))))
                    if not valid_image_mask[gy_center, gx_center]:
                        continue

                    gx1, gy1, gx2, gy2 = bx1 + x0, by1 + y0, bx2 + x0, by2 + y0
                    rows.append({
                        "image": image_name, "tile_size": tile_size,
                        "tile_id": f"{tile_size}_{tile_number}",
                        "tile_x": x0, "tile_y": y0, "confidence": score,
                        "x1": gx1, "y1": gy1, "x2": gx2, "y2": gy2,
                        "center_x_px": (gx1 + gx2) / 2,
                        "center_y_px": (gy1 + gy2) / 2,
                        "width_px": gx2 - gx1, "height_px": gy2 - gy1,
                        "approx_diameter_px": ((gx2 - gx1) + (gy2 - gy1)) / 2,
                        "tile_stretch_low": stretch_low,
                        "tile_stretch_high": stretch_high
                    })
    TILE_DIAGNOSTICS[image_name] = {
        "total_tiles_generated": total_tiles,
        "tiles_skipped_under_10_valid_pixels": skipped_sparse_tiles,
        "tiles_sent_to_model": total_tiles - skipped_sparse_tiles,
        "source_finite_pixels": image_valid_count,
        "source_total_pixels": int(gray.size),
    }
    return rows, total_tiles


print("Tile inference functions ready. Sparse/NoData-heavy tiles will be skipped, not fatal.")

Tile inference functions ready. Sparse/NoData-heavy tiles will be skipped, not fatal.


## 6. Merge duplicate tile predictions safely
Overlapping tiles produce duplicate detections. The merge below suppresses a prediction only when it has both substantial IoU overlap and a similar diameter. This prevents a single oversized box from automatically suppressing several much smaller crater candidates. The raw, pre-merge tile predictions are also exported for auditing.

In [16]:
def box_iou(a,b):
    ix1,iy1=max(a["x1"],b["x1"]),max(a["y1"],b["y1"])
    ix2,iy2=min(a["x2"],b["x2"]),min(a["y2"],b["y2"])
    iw,ih=max(0.0,ix2-ix1),max(0.0,iy2-iy1)
    inter=iw*ih
    area_a=max(0.0,a["width_px"])*max(0.0,a["height_px"])
    area_b=max(0.0,b["width_px"])*max(0.0,b["height_px"])
    return inter/max(area_a+area_b-inter,1e-9)


def scale_aware_merge(rows, iou_threshold=0.55, diameter_ratio_limit=1.8):
    if not rows: return []
    ordered=sorted(rows,key=lambda r:r["confidence"],reverse=True)
    kept=[]
    for row in ordered:
        suppress=False
        d=max(row["approx_diameter_px"],1e-6)
        for prior in kept:
            pd=max(prior["approx_diameter_px"],1e-6)
            ratio=max(d,pd)/min(d,pd)
            if ratio <= diameter_ratio_limit and box_iou(row,prior) >= iou_threshold:
                suppress=True; break
        if not suppress: kept.append(row)
    return kept

print("Scale-aware merge ready.")

Scale-aware merge ready.


## 7. Run inference on every uploaded image
This cell automatically processes every discovered image. Each image is stretched per 256-pixel tile, inferred, mapped to original coordinates, and merged. Errors are recorded per file so one unsupported file does not stop the rest.

In [17]:
# GeoTIFF sanity check: run this cell before inference if a TIFF is being rejected.
# It reports Rasterio's band/mask/NoData interpretation without modifying the source file.
for _p in image_paths:
    if _p.suffix.lower() in {".tif", ".tiff", ".geotiff"}:
        with rasterio.open(_p) as _src:
            _raw = _src.read(TIFF_BAND, masked=False)
            _masked = _src.read(TIFF_BAND, masked=True)
            _mask = np.ma.getmaskarray(_masked)
            _finite_raw = np.isfinite(_raw)
            print(f"\nTIFF: {_p.name}")
            print("shape/dtype:", _raw.shape, _raw.dtype, "bands:", _src.count)
            print("Rasterio nodata:", _src.nodata, "mask flags:", _src.mask_flag_enums)
            print("raw finite:", int(_finite_raw.sum()), "/", _raw.size)
            print("masked pixels:", int(_mask.sum()), "/", _raw.size)
            print("valid after Rasterio mask + finite check:", int((_finite_raw & ~_mask).sum()))
            if _finite_raw.any():
                _v = _raw[_finite_raw]
                print("raw min / p2 / median / p98 / max:",
                      np.percentile(_v, [0, 2, 50, 98, 100]).tolist())



TIFF: nac_crop_georef.tif
shape/dtype: (2048, 2048) uint8 bands: 1
Rasterio nodata: 0.0 mask flags: ([<MaskFlags.nodata: 8>],)
raw finite: 4194304 / 4194304
masked pixels: 1721204 / 4194304
valid after Rasterio mask + finite check: 2473100
raw min / p2 / median / p98 / max: [0.0, 0.0, 46.0, 164.0, 255.0]


In [ ]:
all_tile_candidate_rows=[]
all_candidate_rows=[]
image_summaries=[]
inference_errors=[]
image_geospatial_rows=[]

for image_path in tqdm(image_paths, desc="Processing lunar images"):
    try:
        original = load_grayscale_image(image_path)
        h,w = original.shape
        raw_rows, n_tiles = predict_image_tiles(original, image_path.name)
        geo_meta = GEOSPATIAL_METADATA.get(str(image_path), {})
        if geo_meta:
            image_geospatial_rows.append({"image": image_path.name, "image_path": str(image_path), **geo_meta})
        merged_rows = scale_aware_merge(raw_rows)
        for i,row in enumerate(raw_rows,1):
            row["tile_candidate_id"] = i
            all_tile_candidate_rows.append({"image_path":str(image_path), **row})
        for i,row in enumerate(merged_rows,1):
            row["candidate_id"] = i
            row["image_path"] = str(image_path)
            all_candidate_rows.append(row)
        image_summaries.append({
            "image":image_path.name,"image_path":str(image_path),
            "width_px":w,"height_px":h,"tiles_processed":n_tiles,
            "source_finite_pixels": TILE_DIAGNOSTICS.get(image_path.name, {}).get("source_finite_pixels"),
            "tiles_skipped_under_10_valid_pixels": TILE_DIAGNOSTICS.get(image_path.name, {}).get("tiles_skipped_under_10_valid_pixels", 0),
            "tiles_sent_to_model": TILE_DIAGNOSTICS.get(image_path.name, {}).get("tiles_sent_to_model", n_tiles),
            "raw_tile_predictions":len(raw_rows),"merged_candidates":len(merged_rows),
            "crs": geo_meta.get("crs"), "pixel_resolution_x": (geo_meta.get("resolution") or [None, None])[0],
            "pixel_resolution_y": (geo_meta.get("resolution") or [None, None])[1],
            "nodata_value": geo_meta.get("nodata"),
            **{f"count_conf_{t:.2f}":int(sum(r["confidence"]>=t for r in merged_rows)) for t in THRESHOLDS_TO_REPORT}
        })
        diag = TILE_DIAGNOSTICS.get(image_path.name, {})
        print(f"{image_path.name}: {w}x{h} | finite_pixels={diag.get('source_finite_pixels')} "
              f"| tiles={n_tiles} | skipped_sparse={diag.get('tiles_skipped_under_10_valid_pixels', 0)} "
              f"| sent_to_model={diag.get('tiles_sent_to_model', n_tiles)} "
              f"| raw={len(raw_rows)} | merged={len(merged_rows)}")
    except Exception as e:
        inference_errors.append({"image":str(image_path),"error":repr(e)})
        print("ERROR:",image_path.name,repr(e))

raw_tile_df = pd.DataFrame(all_tile_candidate_rows)
candidate_df = pd.DataFrame(all_candidate_rows)
summary_df = pd.DataFrame(image_summaries)
errors_df = pd.DataFrame(inference_errors)
geospatial_metadata_df = pd.DataFrame(image_geospatial_rows)

detections_df = candidate_df[candidate_df.confidence >= DISPLAY_CONFIDENCE].copy() if not candidate_df.empty else candidate_df.copy()
print("\nImages processed:",len(summary_df),"| errors:",len(errors_df))
print("Raw tile predictions:",len(raw_tile_df),"| merged candidates:",len(candidate_df),"| displayed:",len(detections_df))
display(summary_df)
if not errors_df.empty: display(errors_df)

Processing lunar images:   0%|          | 0/1 [00:00<?, ?it/s]

## 8. Draw annotated images and create threshold comparisons
All boxes are drawn in the original image coordinate system. The annotated image uses the same whole-image display stretch for readability; inference itself used the training-style per-tile stretch.

In [ ]:
annotated_paths = []
drawing_errors = []

for image_path in tqdm(image_paths, desc="Drawing results"):
    if summary_df.empty or not (summary_df.image_path == str(image_path)).any():
        continue
    try:
        original = load_grayscale_image(image_path)
        display_img, _, _ = stretch_training_style(original)
        canvas = cv2.cvtColor(display_img, cv2.COLOR_GRAY2BGR)
        rows = candidate_df[candidate_df.image_path == str(image_path)] if not candidate_df.empty else pd.DataFrame()

        drawn_count = 0
        for _, r in rows.iterrows():
            if float(r.confidence) < DISPLAY_CONFIDENCE:
                continue
            x1, y1, x2, y2 = [int(round(r[k])) for k in ["x1", "y1", "x2", "y2"]]
            x1 = max(0, min(x1, canvas.shape[1] - 1))
            x2 = max(0, min(x2, canvas.shape[1] - 1))
            y1 = max(0, min(y1, canvas.shape[0] - 1))
            y2 = max(0, min(y2, canvas.shape[0] - 1))
            if x2 <= x1 or y2 <= y1:
                continue
            cv2.rectangle(canvas, (x1, y1), (x2, y2), (0, 255, 0), 1)
            cv2.putText(canvas, f"{r.confidence:.2f}", (x1, max(14, y1 - 3)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 0), 1, cv2.LINE_AA)
            drawn_count += 1

        out = ANNOTATED_DIR / f"{image_path.stem}_detections.jpg"
        write_ok = cv2.imwrite(str(out), canvas)
        if not write_ok or not out.exists() or out.stat().st_size == 0:
            raise IOError(f"OpenCV failed to write annotated image: {out}")
        annotated_paths.append(str(out))
        print(f"Saved {out.name} | boxes drawn at conf >= {DISPLAY_CONFIDENCE}: {drawn_count}")
    except Exception as e:
        drawing_errors.append({"image": str(image_path), "error": repr(e)})
        print("Drawing error:", image_path.name, repr(e))

threshold_report = pd.DataFrame([
    {"confidence_threshold": t,
     "total_predictions": int((candidate_df.confidence >= t).sum()) if not candidate_df.empty else 0}
    for t in THRESHOLDS_TO_REPORT
])
print("Threshold comparison:")
display(threshold_report)
print("Annotated files:", len(annotated_paths), "| drawing errors:", len(drawing_errors))
if drawing_errors:
    display(pd.DataFrame(drawing_errors))

## 9. Inspect annotated results
The plot below displays up to eight annotated images. If an image is very large, the notebook still preserves its original dimensions in the saved output.

In [ ]:
if not annotated_paths:
    print("No annotated images were generated.")
    if not summary_df.empty:
        print("Images completed inference, but check the drawing errors above and the annotated_images output folder.")
    elif not errors_df.empty:
        print("Inference failed for the discovered images. Review processing_errors.csv / errors_df above.")
else:
    n_show = min(8, len(annotated_paths))
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 6 * n_show), squeeze=False)
    for ax, image_file in zip(axes.ravel(), annotated_paths[:n_show]):
        img = cv2.imread(image_file, cv2.IMREAD_COLOR)
        if img is None:
            ax.set_title(f"Could not read: {Path(image_file).name}")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(Path(image_file).name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 10. Export all diagnostic files

The notebook writes its CSV tables, run metadata, geospatial metadata, annotated images and a compressed ZIP archive to `/kaggle/working/crater_detection_results`. The ZIP is convenient for downloading the complete run from Kaggle.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tabular outputs
raw_tile_df.to_csv(OUTPUT_DIR / "raw_tile_predictions.csv", index=False)
candidate_df.to_csv(OUTPUT_DIR / "all_candidates_conf_0.001.csv", index=False)
detections_df.to_csv(OUTPUT_DIR / "detections_at_display_threshold.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "image_summary.csv", index=False)
threshold_report.to_csv(OUTPUT_DIR / "threshold_comparison.csv", index=False)
errors_df.to_csv(OUTPUT_DIR / "processing_errors.csv", index=False)
if "drawing_errors" in globals():
    pd.DataFrame(drawing_errors).to_csv(OUTPUT_DIR / "drawing_errors.csv", index=False)
if "geospatial_metadata_df" in globals():
    geospatial_metadata_df.to_csv(OUTPUT_DIR / "geospatial_metadata.csv", index=False)

metadata = {
    "task": "lunar crater detection",
    "checkpoint": str(MODEL_PATH), "model_type": MODEL_TYPE, "classes": model.names,
    "training_matched_source_tile_size": 256, "source_tile_scales": SOURCE_TILE_SIZES,
    "yolo_imgsz": IMAGE_SIZE, "tile_overlap": TILE_OVERLAP,
    "low_confidence_cache": LOW_CONFIDENCE_CACHE,
    "validation_selected_confidence": VALIDATION_CONFIDENCE,
    "exploratory_display_confidence": DISPLAY_CONFIDENCE,
    "merge_iou_threshold": 0.55, "merge_diameter_ratio_limit": 1.8,
    "input_search_location": discovered_from, "images_found": len(image_paths),
    "images_processed": len(summary_df), "raw_tile_predictions": len(raw_tile_df),
    "merged_candidates": len(candidate_df), "displayed_detections": len(detections_df),
    "errors": len(errors_df), "python": platform.python_version(),
    "ultralytics": __import__("ultralytics").__version__,
    "rasterio": rasterio.__version__, "tiff_reader": "rasterio",
    "tiff_band_1based": TIFF_BAND, "tiff_nodata_override": TIFF_NODATA_OVERRIDE,
    "geotiff_metadata_images": len(GEOSPATIAL_METADATA),
    "no_data_center_detections_filtered": True
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2, default=str))
(OUTPUT_DIR / "geospatial_metadata.json").write_text(
    json.dumps(GEOSPATIAL_METADATA, indent=2, default=str)
)
(OUTPUT_DIR / "result_summary.txt").write_text("\n".join([
    "LUNAR CRATER DETECTION SUMMARY",
    f"Checkpoint: {MODEL_PATH}", f"Model type: {MODEL_TYPE}",
    f"Images found: {len(image_paths)}", f"Images processed: {len(summary_df)}",
    f"Raw tile predictions: {len(raw_tile_df)}", f"Merged candidates: {len(candidate_df)}",
    f"Displayed detections (conf >= {DISPLAY_CONFIDENCE}): {len(detections_df)}",
    f"Errors: {len(errors_df)}", f"GeoTIFF metadata records: {len(GEOSPATIAL_METADATA)}",
    "Threshold counts:",
    *[f" >= {t:.2f}: {int((candidate_df.confidence>=t).sum()) if not candidate_df.empty else 0}" for t in THRESHOLDS_TO_REPORT]
]), encoding="utf-8")

# Zip all outputs, including annotated images and the metadata files.
zip_path = Path("/kaggle/working/lunar_crater_detection_results.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file() and file_path.resolve() != zip_path.resolve():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_DIR))

with zipfile.ZipFile(zip_path, "r") as zf:
    bad_member = zf.testzip()
    members = zf.namelist()
if bad_member:
    raise RuntimeError(f"ZIP verification failed at {bad_member}")

print("\nSaved outputs to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_DIR), f"({p.stat().st_size/1024:.1f} KB)")
print(f"\nZIP archive: {zip_path} | {len(members)} files | {zip_path.stat().st_size/1024/1024:.2f} MB")
print("ZIP integrity check: PASSED")

## 11. How to use this notebook

1. Attach the crater checkpoint dataset in Kaggle Add Input. The notebook prefers `best_finetuned_crater_model.pt`; the baseline is a fallback.
2. Upload image files to `/kaggle/working/exam_images`, or attach a Kaggle image dataset. The notebook searches automatically if that folder is empty.
3. Run all cells from top to bottom.
4. For multi-band TIFF/GeoTIFF files, confirm `TIFF_BAND` in the loader cell. Band 1 is the default.
5. Keep `TIFF_NODATA_OVERRIDE = None` unless the product documentation explicitly gives a NoData value.
6. Download `lunar_crater_detection_results.zip` from Kaggle's `/kaggle/working` output, or download the `crater_detection_results` folder.

### Interpretation limits
- The detector returns model-predicted bounding boxes, not verified ground-truth craters.
- The approximate crater diameter is derived from the predicted bounding box in pixels.
- GeoTIFF CRS/transform/resolution are preserved as metadata, but this notebook does not yet convert every crater detection into map coordinates or calculate physical crater diameters.
- A pixel resolution alone may not represent uniform lunar surface distance if the image is not an appropriate map projection.
- The 0.15 display threshold is exploratory; use labelled evaluation data to select a final threshold.